In [43]:
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

In [44]:
def drop_desc(df):
    ''' Elimina le colonne descrittive.
        Parametri: 
            df: DataFrame da elaborare 
        Restituisce il DataFrame elaborato
    '''
    desc_cols = [c for c in df.columns if c.startswith("DESC_")]
    return df.drop(columns=desc_cols, errors="ignore")

In [45]:
def clean_order_date(df, col, min_date, max_date):
    ''' Converte le colonne in formato leggibile e sostituisce i valori fuori dall' intervallo (min_date,max_date)
        con la mediana delle date.
        Parametri:
            df: DataFrame da elaborare
            col: colonne da elaborare (colonne con date)
            min_date,max_date: data minima e massima
        Restituisce:
            d: DataFrame elaborato
    '''
    d = pd.to_datetime(df[col], format="%Y%m%d", errors="coerce")
    mask = (d < min_date) | (d > max_date)
    d[mask] = pd.NaT
    d = d.fillna(d.median())
    return d

In [46]:
def clean_invoice_date(df, invoice_col, order_col, min_date, max_date):
    '''
    Converte la colonna invoice date e sostituisce i valori fuori intervallo
    o null con la corrispondente order date.

    Parametri:
        df: DataFrame da elaborare
        invoice_col: colonna delle invoice date
        order_col: colonna delle order date (fallback)
        min_date, max_date: intervallo valido

    Restituisce:
        d: Series pulita delle invoice date
    '''
    
    # Convert both columns to datetime
    invoice = pd.to_datetime(df[invoice_col], format="%Y%m%d", errors="coerce")
    order = pd.to_datetime(df[order_col], format="%Y%m%d", errors="coerce")
    
    # Mask for invalid invoice dates
    mask = (invoice < min_date) | (invoice > max_date)
    invoice[mask] = pd.NaT
    
    # Replace NaT with corresponding order date
    invoice = invoice.fillna(order)
    
    return invoice

# -----------------------------
# 1. CARICAMENTO E MERGE DATI
# -----------------------------

In [47]:
df_sales = pd.read_csv("SALES.csv")
df_company = pd.read_csv("COMPANY_LOOKUP.csv").drop_duplicates()
df_items = pd.read_csv("ITEM_LOOKUP.csv").drop_duplicates()
df_lines = pd.read_csv("ITEM_BUSINESS_LINE_LOOKUP.csv").drop_duplicates()
df_customers = pd.read_csv("CUSTOMER_LOOKUP.csv").drop_duplicates()
df_area = pd.read_csv("AREA_MANAGER_LOOKUP.csv").drop_duplicates()

df = df_sales.merge(df_company, on="ID_COMPANY", how="left")
df = df.merge(df_items, on="IDS_ITEM", how="left")
df = df.merge(df_lines, on="ID_BUSINESS_LINE", how="left")
df = df.merge(df_customers, on="IDS_CUSTOMER", how="left")
df = df.merge(df_area, on="ID_AREA_MANAGER", how="left")

In [48]:
# Conteggio dei valori NULL
print("VALORI MANCANTI:")
for col in df.columns:
    n_nans = df[col].isna().sum()
    print(col, n_nans)

VALORI MANCANTI:
ID_COMPANY 0
ID_ORDER_NUM 0
IDS_CUSTOMER 0
IDS_ITEM 0
ID_ORDER_DATE 0
ID_INVOICE_DATE 0
VAL_REVENUES 0
VAL_COST 0
DESC_COMPANY 0
DESC_ITEM 0
ID_BUSINESS_LINE 0
DESC_BUSINESS_LINE 0
DESC_CUSTOMER 0
ID_COUNTRY 0
ID_AREA_MANAGER 0
DESC_AREA_MANAGER 0


# -----------------------------
# 2. PULIZIA DATE
# -----------------------------

In [49]:
min_date = pd.Timestamp("1900-01-01")
max_date = pd.Timestamp("today")

df["ID_ORDER_DATE"] = clean_order_date(df, "ID_ORDER_DATE",   min_date, max_date)
df["ID_INVOICE_DATE"] = clean_invoice_date(df, "ID_INVOICE_DATE","ID_ORDER_DATE", min_date, max_date)


In [50]:
df.to_csv("SALES_OLAP.csv",index=False)
print(df)

     ID_COMPANY  ID_ORDER_NUM   IDS_CUSTOMER         IDS_ITEM ID_ORDER_DATE  \
0            40      25437824   00040-456235   7273238-405MAG    2025-05-15   
1            40      25428861   00040-456235   7273238-405MAG    2025-03-31   
2            40      25473527   00040-456235   7273238-405MAG    2025-11-17   
3            40      26422305   00040-456235   7479346-405MAG    2026-03-02   
4         80640      25400075  80640-1048669  7278569-W401MAG    2025-01-10   
..          ...           ...            ...              ...           ...   
219       80640      26400184   80640-518561  7344923-W401MAG    2026-02-04   
220          40      25467026   00040-391550   7481581-405MAG    2025-10-17   
221       80640      25400237   80640-414346  7304109-W401MAG    2025-02-06   
222       80640      25401171   80640-314309  7344923-W401MAG    2025-06-18   
223       80640      25401375   80640-314309  7245498-W401MAG    2025-07-23   

    ID_INVOICE_DATE  VAL_REVENUES    VAL_COST   DES